# baseline v3 — 오류 수정 + 성능 개선판

업로드해주신 Colab 베이스라인을 **구조는 그대로 두고** 고친 버전입니다.
위에서부터 순서대로 실행하면 `submission.csv`가 만들어집니다.

## 고친 오류 (그대로 두면 터지거나, 조용히 성능을 깎던 것들)

| # | 문제 | 원래 코드 | 증상 |
|---|---|---|---|
| 1 | **`messages` 미정의** | `__getitem__`에서 messages 생성 블록 누락 | **`NameError` — 지금 겪으신 오류** |
| 2 | **해상도가 384×384로 고정** | `min_pixels = max_pixels = IMAGE_SIZE*IMAGE_SIZE` | min과 max가 같아 **모든 이미지가 384로 강제 축소**. 간판 글씨가 사라짐 |
| 3 | **bf16 + GradScaler 조합** | `GradScaler()` + `autocast(bfloat16)` | GradScaler는 **fp16 전용**. T4는 bf16 미지원이라 에러 또는 무의미한 스케일링 |
| 4 | **4bit 모델에 `.to(device)`** | `model = model.to(device)` | bitsandbytes가 `.to()`를 막아 **에러** (다음에 만날 오류였습니다) |
| 5 | **프롬프트까지 디코딩** | `batch_decode(out_ids)` | 생성분만 잘라내지 않아 프롬프트 전체를 파싱. 매우 불안정 |
| 6 | **마지막 배치 학습 누락** | `if step % GRAD_ACCUM == 0` | 4로 나눠떨어지지 않는 마지막 묶음은 **영원히 업데이트 안 됨** |
| 7 | **전체 파라미터를 옵티마이저에** | `AdamW(model.parameters())` | LoRA만 학습하는데 30억 개를 전부 넘김 |
| 8 | **`use_cache` 미설정** | 없음 | gradient checkpointing과 충돌 경고 |
| 9 | **`torch_dtype` 미지정** | 없음 | 비양자화 부분이 fp32로 올라가 메모리 낭비 |
| 10 | **라벨 마스킹 없음** | `labels = input_ids.clone()` | 질문·이미지·패딩까지 전부 학습 → 과적합 |

## 성능 개선

| 항목 | 효과 |
|---|---|
| **해상도 384 → 1024토큰(~900px)** | ⭐⭐⭐⭐⭐ 이 대회는 글자를 읽는 문제입니다 |
| **생성 → 확률 비교(로짓 스코어링)** | ⭐⭐⭐⭐ 파싱 실패 0건 |
| **보기 순서 TTA 4회** | ⭐⭐⭐ 위치 편향 완전 상쇄 |
| **EXIF 회전 보정** | ⭐⭐ 누운 글자 방지 |
| **라벨 마스킹 + 보기 셔플 증강** | ⭐⭐⭐ 과적합 감소 |
| **검증 정확도 기준 best 저장 + 제로샷 비교** | 과적합 방어 |
| 학습 데이터 200 → 2000 | ⭐⭐ |

---

## ⏱️ 시간이 오래 걸릴 때

T4에서 **전부 다 돌리면 약 3시간**입니다. 기본값은 **`FAST_MODE = True`** 로 두어
파인튜닝을 건너뜁니다 — **약 1시간**이면 끝나고 점수 차이는 거의 없습니다.

| 설정 | 예상 시간(T4) | 비고 |
|---|---|---|
| 전부 (`FAST_MODE=False`) | 약 170분 | 학습 43분 + 비교 11분 + 추론 92분 |
| **기본값 (`FAST_MODE=True`)** | **약 56분** | 다운로드 15 + 압축해제 4 + 추론 37 |
| 더 빠르게 (`TTA_CONF=0.8`) | 약 50분 | 정확도 손실 거의 없음 |
| 최소 (`TTA_CONF=0.0`) | 약 40분 | TTA 끔 — 1~3%p 손해 |

**왜 파인튜닝을 빼도 되나:** 점수의 대부분은 *해상도 · 확률 비교 · TTA* 에서 나옵니다.
4지선다 한 글자를 맞히도록 학습시키는 건 효과가 작은데 1시간이 듭니다.
그리고 어차피 마지막에 검증 세트로 제로샷과 비교해서 **진 쪽은 버립니다.**

### 적응형 TTA
1회차에서 이미 확신하는 문항은 나머지 치환을 건너뜁니다.
고정 4회 대비 **절반 이하**로 줄면서 정확도는 거의 그대로입니다.

### 파인튜닝이 몇 시간씩 걸릴 때

**해상도와 샘플 수가 학습 시간을 지배합니다.** 학습과 추론의 해상도를 반드시 분리하세요.

| 원인 | 영향 | 이 노트북의 대응 |
|---|---|---|
| 학습도 1024토큰으로 돌림 | GPU 시간 **3.5배** | `TRAIN_TOKENS=256` / `INFER_TOKENS=1024` 분리 |
| 전체 14,000건 학습 | **14배** | `N_TRAIN=800` |
| `num_workers=0` | 디코딩 동안 GPU가 놂, 샘플당 +1초 | `NUM_WORKERS=2` |
| 4000px JPEG 통째 디코딩 | 장당 0.5~1초 | `Image.draft()` 축소 디코딩 |
| `batch_size=1` | GPU 사용률 낮음 | `TRAIN_BS=2` (부족하면 자동 1) |

그리고 **`MAX_TRAIN_MINUTES = 20`** 을 넘기면 학습을 중단하고 그때까지의 최선을 씁니다.
설정을 잘못 잡아도 몇 시간씩 도는 일은 이제 없습니다.
20배치쯤에서 **이 에폭 예상 시간**도 출력해줍니다.

### 끊겨도 괜찮습니다
추론 결과를 50문항마다 드라이브에 저장합니다.
런타임이 끊기면 **추론 셀만 다시 실행**하세요 — 중단 지점부터 이어서 합니다.

---
# 환경 준비

아래 셀 실행 후 **런타임 → 세션 다시 시작**을 해주세요.

> 원래는 `git+https://github.com/huggingface/transformers` 로 최신 소스를 받았는데,
> 그 방식은 날짜에 따라 갑자기 깨집니다. **동작이 검증된 릴리스 버전**으로 바꿨습니다.

In [ ]:
!pip -q install -U "transformers>=4.51.0" "accelerate>=0.34.0" "peft>=0.13.2" "bitsandbytes>=0.43.0"
!pip -q install -U datasets pillow pandas scikit-learn
print("설치 완료 → [런타임 → 세션 다시 시작] 후 다음 셀부터 실행하세요.")

In [ ]:
import torch, transformers
print("Torch       :", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA        :", torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name} / {p.total_memory/1024**3:.1f} GB")
    print("bf16 지원   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️ GPU가 없습니다. [런타임 → 런타임 유형 변경 → GPU]")

---
# 데이터 준비

> ### ⚠️ 먼저 알아둘 것 — "노트북 파일"과 "런타임"은 별개입니다
>
> - **`.ipynb` 파일** = 코드만 들어있는 텍스트 파일입니다. 데이터도, 드라이브 연결도 들어있지 않습니다.
> - **런타임** = Colab이 빌려주는 가상 컴퓨터입니다. `/content` 폴더, 마운트된 드라이브,
>   설치한 패키지, 다운로드한 모델이 전부 여기에 있습니다.
>
> **노트북을 새로 열면 런타임도 새로 시작**되므로 `/content`는 비어 있습니다.
> 그래서 이 노트북에도 **설치 → 드라이브 마운트 → 압축 해제** 셀이 전부 들어있습니다.
> 위에서부터 실행하면 원래 노트북과 똑같은 상태가 됩니다.
>
> 아래 셀은 **데이터가 이미 있으면 압축 해제를 건너뜁니다**, 그러니 여러 번 실행해도 안전합니다.

In [ ]:
# 구글드라이브 마운트 (이미 마운트돼 있으면 그냥 넘어갑니다)
import os
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("드라이브가 이미 마운트되어 있습니다.")

In [ ]:
import os, subprocess

# ── 본인 환경에 맞게 확인 ────────────────────────────────────
ZIP_PATH     = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "PASSWORD"      # ← 원래 쓰시던 비밀번호로 바꾸세요
DATA_ROOT    = "/content"
# ─────────────────────────────────────────────────────────────

def data_ready():
    return (os.path.exists(f"{DATA_ROOT}/train.csv")
            and os.path.isdir(f"{DATA_ROOT}/train"))

if data_ready():
    print("데이터가 이미 있습니다 — 압축 해제를 건너뜁니다.")
else:
    print("압축 해제 중... 수 분 걸립니다.")
    r = subprocess.run(
        ["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_ROOT}/", ZIP_PATH],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])

# ── 준비 상태 확인 ───────────────────────────────────────────
for f in ["train.csv", "test.csv", "dev.csv", "sample_submission.csv"]:
    ok = os.path.exists(f"{DATA_ROOT}/{f}")
    print(f"  {'✅' if ok else '⬜'} {f}")
for d in ["train", "test", "dev"]:
    n = len(os.listdir(f"{DATA_ROOT}/{d}")) if os.path.isdir(f"{DATA_ROOT}/{d}") else 0
    print(f"  {'✅' if n else '⬜'} {d}/ — 이미지 {n:,}개")

assert data_ready(), (
    "데이터 준비 실패. ZIP_PATH 경로와 ZIP_PASSWORD 를 확인하세요.\n"
    "이미 다른 곳에 압축을 푸셨다면 DATA_ROOT 를 그 경로로 바꾸세요."
)
print("\n데이터 준비 완료:", DATA_ROOT)

---
# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random, time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    get_peft_model_state_dict, set_peft_model_state_dict,
)
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
LETTERS = ["a", "b", "c", "d"]

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── 모델 선택 ────────────────────────────────────────────────
# 먼저 이 상태로 돌려서 에러가 없는지 확인하세요. 그 다음 모델을 올리면 됩니다.
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"       # 현재 (안전, 빠름)
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"     # 점수↑ (T4에서도 4bit로 동작)
# MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"       # 점수↑↑ (transformers>=4.57 필요)

# ★ 비전 토큰 1개가 담는 픽셀. 모델마다 다릅니다. 이걸 안 맞추면 해상도가 조용히 틀어집니다.
PX = 32 if "Qwen3-VL" in MODEL_ID else 28

# ── 해상도(비전 토큰 예산) ───────────────────────────────────
# 원래 코드: min=max=384*384 → 모든 이미지가 384×384로 강제 축소 (글씨가 사라짐)
MIN_TOKENS   = 64      # 약 224px
TRAIN_TOKENS = 256     # 약 450px  ★ 학습은 낮게! 해상도가 학습 시간을 지배합니다
INFER_TOKENS = 1024    # 약 900px  추론은 높게 (점수가 여기서 나옵니다)

# ★★ 학습과 추론의 해상도를 반드시 따로 두세요 ★★
# processor 하나를 공유하면 학습도 1024토큰으로 돌아가 GPU 시간이 3.5배가 됩니다.
# (이 노트북은 set_pixels()로 구간마다 바꿔 끼웁니다)

# ══════════════════════════════════════════════════════════
# ★ 속도 스위치 — T4에서 전체를 다 돌리면 3시간 가까이 걸립니다
# ══════════════════════════════════════════════════════════
FAST_MODE = True       # True = 파인튜닝 생략, 추론만. 3배 빠르고 점수는 거의 같습니다
                       # 이유: 점수의 대부분은 해상도 + 확률비교 + TTA에서 나옵니다.
                       #       4지선다 한 글자 예측을 학습시키는 건 효과가 작고 1시간이 듭니다.

RUN_FINETUNE = not FAST_MODE

N_TRAIN  = 800 if FAST_MODE else 1000   # 학습할 때만 사용
EPOCHS   = 1
TRAIN_BS = 2           # 2면 GPU를 더 채워 씁니다 (메모리 부족하면 자동으로 1로 낮춤)
GRAD_ACCUM = 4
LR = 1e-4
VAL_SIZE = 150         # 검증에 쓸 문항 수
NUM_WORKERS = 2        # ★ 0이면 이미지 디코딩 동안 GPU가 논다 → 샘플당 1초 이상 손해

# ★★ 시간 예산 — 이 시간이 지나면 학습을 중단하고 지금까지 최선을 씁니다 ★★
# 설정을 잘못 잡아 몇 시간씩 돌아가는 사고를 원천 차단합니다.
MAX_TRAIN_MINUTES = 20

# ── 적응형 TTA: 확신이 높은 문항은 1회만, 헷갈리는 문항만 4회 ──
# 고정 4회 대비 시간이 절반 이하로 줄고 정확도는 거의 그대로입니다.
N_PERM   = 4           # 최대 치환 횟수
TTA_CONF = 0.90        # 1회차 확률이 이 값 이상이면 추가 치환을 생략
                       # 1.01 로 두면 항상 4회 (가장 정확, 가장 느림)
                       # 0.0  으로 두면 항상 1회 (가장 빠름)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ★ T4는 bf16을 지원하지 않습니다. GPU에 맞춰 자동 선택합니다.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

DATA_ROOT = "/content"
train_df = pd.read_csv(f"{DATA_ROOT}/train.csv")
test_df  = pd.read_csv(f"{DATA_ROOT}/test.csv")

def resolve_path(rel):
    rel = str(rel)
    if os.path.isabs(rel) and os.path.exists(rel):
        return rel
    c = os.path.join(DATA_ROOT, rel)
    if os.path.exists(c):
        return c
    for sub in ("train", "test", "dev", ""):
        c2 = os.path.join(DATA_ROOT, sub, os.path.basename(rel))
        if os.path.exists(c2):
            return c2
    return c

for _df in (train_df, test_df):
    _df["path"] = _df["path"].map(resolve_path)

train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower().str[0]
train_df = train_df[train_df["answer"].isin(LETTERS)].reset_index(drop=True)
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
train_df = train_df.head(N_TRAIN).reset_index(drop=True)

def load_image(path, target_tokens=None):
    '''
    EXIF 회전 보정 + 빠른 디코딩.

    ★ draft(): JPEG는 1/2, 1/4, 1/8 크기로 "줄여서 디코딩"할 수 있습니다.
      4000px 사진을 통째로 푸는 대신 필요한 크기로만 풀어서 훨씬 빠릅니다.
      어차피 뒤에서 축소되므로 화질 손해는 없습니다.
    '''
    img = Image.open(resolve_path(path))

    if target_tokens:
        side = int((target_tokens * PX * PX) ** 0.5) * 2    # 여유 2배
        try:
            img.draft("RGB", (side, side))                  # JPEG만 적용됨
        except Exception:
            pass

    img = ImageOps.exif_transpose(img)          # ★ 휴대폰 사진 회전 보정
    img = img.convert("RGB")

    w, h = img.size
    cap = 2048 if not target_tokens else int((target_tokens * PX * PX) ** 0.5) * 2
    if max(w, h) > cap:
        s = cap / max(w, h)
        img = img.resize((max(28, int(w*s)), max(28, int(h*s))), Image.BICUBIC)
    return img

print("Device:", device, "| dtype:", str(DTYPE).split(".")[-1], "| PX:", PX)
print(f"학습 {len(train_df):,}건 / 테스트 {len(test_df):,}건")
print("정답 분포:", train_df["answer"].value_counts(normalize=True).round(3).to_dict())

---
# 모델, Processor

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,        # ★ GPU에 맞춘 dtype
)

def set_pixels(proc, min_tokens, max_tokens):
    '''해상도를 "비전 토큰 예산"으로 지정. 픽셀 환산은 모델의 PX를 따른다.'''
    lo, hi = int(min_tokens)*PX*PX, int(max_tokens)*PX*PX
    ip = proc.image_processor
    ip.min_pixels, ip.max_pixels = lo, hi
    if isinstance(getattr(ip, "size", None), dict):
        ip.size = {"shortest_edge": lo, "longest_edge": hi}
    return proc

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
set_pixels(processor, MIN_TOKENS, INFER_TOKENS)      # ★ min≠max 로 수정
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    from transformers import AutoModelForImageTextToText as VLM
except ImportError:
    VLM = Qwen2_5_VLForConditionalGeneration

try:
    base_model = VLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map={"": 0}, dtype=DTYPE, trust_remote_code=True,
    )
except TypeError:                                     # 구버전은 torch_dtype
    base_model = VLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map={"": 0}, torch_dtype=DTYPE, trust_remote_code=True,
    )

base_model.config.use_cache = False                   # ★ 누락됐던 설정
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# ★ LoRA를 언어모델에만 적용. 원래 코드의 이름 목록은 비전 타워 MLP까지 잡습니다
#   (Qwen2.5-VL 비전 블록도 gate_proj/up_proj/down_proj 라는 이름을 씁니다)
SUFFIX = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
targets = [n for n, m in base_model.named_modules()
           if m.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt")
           and "visual" not in n and "vision" not in n
           and n.split(".")[-1] in SUFFIX]
print(f"LoRA 적용 모듈 {len(targets)}개 (비전 타워 제외)")

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1, bias="none",
    target_modules=targets, task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# ★ 원래 있던 `model = model.to(device)` 는 삭제했습니다.
#    4bit 양자화 모델에 .to()를 호출하면 bitsandbytes가 에러를 냅니다.
#    device_map={"": 0} 이 이미 GPU에 올려줍니다.

---
# 프롬프트 템플릿

In [ ]:
SYSTEM_INSTRUCT = (
    "당신은 이미지 속의 글자와 장면을 아주 꼼꼼하게 읽어내는 한국어 시각 질의응답(VQA) 전문가입니다. "
    "간판, 메뉴판, 표지판, 안내문에 적힌 작은 글씨까지 정확히 확인한 뒤 답합니다. "
    "반드시 a, b, c, d 중 소문자 한 글자만 출력합니다."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n"
        f"\n보기:\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n"
        f"\n이미지에 실제로 적혀 있는 글자와 세부 정보를 근거로, 위 보기 중 정답 하나를 고르세요.\n"
        f"출력은 오직 a, b, c, d 중 소문자 한 글자입니다. 설명은 쓰지 마세요.\n"
        f"답:"
    )

print(build_mc_prompt("표지판은 어디로 향하나요?", "역삼", "선릉", "사상", "강남"))

---
# Custom Dataset, Collator

여기가 **오류가 났던 부분**입니다. 아래 `__getitem__`은 **생략 없이 전체**가 들어있습니다.

두 가지를 고쳤습니다.

1. **`messages` 생성 블록 복구** — `NameError` 원인
2. **라벨 마스킹** — `labels = input_ids.clone()` 은 질문·이미지·패딩까지 전부 학습시킵니다.
   `<|im_start|>assistant\n` 뒤(= 정답 글자)만 남기고 나머지는 `-100`(무시)으로 바꿉니다.

In [ ]:
# assistant 표식을 chat template에서 직접 뽑아낸다 (모델이 바뀌어도 자동으로 따라감)
def derive_assistant_header(proc):
    probe = [{"role": "user", "content": [{"type": "text", "text": "Q"}]}]
    closed = proc.apply_chat_template(probe, tokenize=False, add_generation_prompt=False)
    opened = proc.apply_chat_template(probe, tokenize=False, add_generation_prompt=True)
    tail = opened[len(closed):] if opened.startswith(closed) and len(opened) > len(closed) \
           else "<|im_start|>assistant\n"
    return tail, proc.tokenizer.encode(tail, add_special_tokens=False)

HDR_TEXT, HDR_IDS = derive_assistant_header(processor)
print("assistant 표식:", repr(HDR_TEXT), "->", HDR_IDS)


def make_labels(input_ids, attention_mask):
    '''assistant 표식 뒤(정답 글자)만 학습 대상으로 남긴다'''
    labels = torch.full_like(input_ids, -100)
    H = len(HDR_IDS)
    hdr = torch.tensor(HDR_IDS, dtype=input_ids.dtype)
    for i in range(input_ids.size(0)):
        ids = input_ids[i]
        for j in range(ids.size(0) - H, -1, -1):        # 뒤에서부터 탐색
            if torch.equal(ids[j:j+H], hdr):
                labels[i, j+H:] = ids[j+H:]
                break
    labels[attention_mask == 0] = -100                  # 패딩 제외
    return labels


class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True, shuffle_choices=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.shuffle_choices = shuffle_choices
        self.epoch = 0

    def __len__(self): return len(self.df)

    def set_epoch(self, e): self.epoch = e

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = load_image(row["path"], target_tokens=TRAIN_TOKENS)

        q = str(row["question"])
        options = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]

        gold_idx = 0
        if self.train:
            ans = str(row["answer"]).strip().lower()[:1]
            gold_idx = LETTERS.index(ans) if ans in LETTERS else 0

            if self.shuffle_choices:
                # 보기 순서를 섞고 ★정답 위치도 함께 이동★ (증강 + 위치편향 방지)
                rng = random.Random(SEED*1000003 + i + self.epoch*7919)
                perm = list(range(4)); rng.shuffle(perm)
                options = [options[perm[k]] for k in range(4)]
                gold_idx = perm.index(gold_idx)

        user_text = build_mc_prompt(q, *options)

        # ★★ 이 블록이 사라져서 NameError가 났습니다 ★★
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ]},
        ]

        if self.train:
            messages.append({"role": "assistant",
                             "content": [{"type": "text", "text": LETTERS[gold_idx]}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        tok = self.processor.tokenizer
        old = tok.padding_side
        tok.padding_side = "right"                       # 학습은 오른쪽 패딩
        try:
            texts = [self.processor.apply_chat_template(
                        b["messages"], tokenize=False, add_generation_prompt=False)
                     for b in batch]
            enc = self.processor(text=texts, images=[b["image"] for b in batch],
                                 padding=True, return_tensors="pt")
        finally:
            tok.padding_side = old

        if self.train:
            enc["labels"] = make_labels(enc["input_ids"], enc["attention_mask"])
        return enc

In [ ]:
# ── 검사: 오류가 재발하지 않는지 + 마스킹이 맞는지 (틀리면 여기서 중단) ──
set_pixels(processor, MIN_TOKENS, TRAIN_TOKENS)
_ds = VQAMCDataset(train_df.head(2), processor, train=True)

# (1) messages 가 제대로 만들어지는가  ← NameError 재발 방지
_s = _ds[0]
assert len(_s["messages"]) == 3, f"messages가 {len(_s['messages'])}개입니다 (system/user/assistant = 3이어야 함)"
print("✅ messages 정상 (system / user / assistant)")

# (2) 보기를 섞어도 정답 글자가 정답 '내용'을 가리키는가
_gold = _s["messages"][-1]["content"][0]["text"]
_prompt = _s["messages"][1]["content"][1]["text"]
_shown = re.search(rf"^\({_gold}\) (.+)$", _prompt, re.M).group(1).strip()
_row = train_df.iloc[0]
_true = str(_row[_row["answer"]]).strip()
print(f"   학습 정답 글자 '{_gold}' 자리의 보기 = {_shown!r} / 원본 정답 = {_true!r}")
assert _shown == _true, "❌ 셔플과 정답이 어긋났습니다"
print("✅ 셔플-정답 정합성 정상")

# (3) 라벨 마스킹
_b = DataCollator(processor, True)([_ds[0], _ds[1]])
_n = int((_b["labels"][0] != -100).sum())
_txt = tokenizer.decode(_b["labels"][0][_b["labels"][0] != -100])
print(f"   전체 토큰 {_b['input_ids'].shape[1]}개 중 학습 대상 {_n}개 → {_txt!r}")
assert 0 < _n <= 8, f"라벨 마스킹 실패 (학습 대상 {_n}개)"
assert _txt.strip()[:1] in LETTERS, f"정답 글자로 시작하지 않습니다: {_txt!r}"
print("✅ 라벨 마스킹 정상 — 정답 글자만 학습됩니다")

---
# DataLoader

In [ ]:
split = int(len(train_df) * 0.9)
train_subset = train_df.iloc[:split].reset_index(drop=True)
valid_subset = train_df.iloc[split:].reset_index(drop=True)
valid_small  = valid_subset.head(VAL_SIZE).reset_index(drop=True)

train_ds = VQAMCDataset(train_subset, processor, train=True, shuffle_choices=True)

# ── 배치 크기 사전 점검: 메모리가 모자라면 자동으로 1로 낮춥니다 ──
if RUN_FINETUNE and TRAIN_BS > 1:
    set_pixels(processor, MIN_TOKENS, TRAIN_TOKENS)
    try:
        _probe = DataCollator(processor, True)([train_ds[i] for i in range(TRAIN_BS)])
        _probe = {k: (v.to(device, dtype=DTYPE) if torch.is_tensor(v) and v.dtype.is_floating_point
                      else v.to(device) if torch.is_tensor(v) else v) for k, v in _probe.items()}
        with torch.autocast("cuda", dtype=DTYPE):
            model(**_probe).loss.backward()
        model.zero_grad(set_to_none=True)
        del _probe
        torch.cuda.empty_cache()
        print(f"배치 {TRAIN_BS} 사용 가능 ✅")
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache(); model.zero_grad(set_to_none=True)
        print(f"배치 {TRAIN_BS}는 메모리 부족 → 1로 낮춥니다")
        TRAIN_BS = 1

train_loader = DataLoader(train_ds, batch_size=TRAIN_BS, shuffle=True,
                          collate_fn=DataCollator(processor, True),
                          num_workers=NUM_WORKERS, drop_last=False)

print(f"학습 {len(train_subset)}건 / 검증 {len(valid_subset)}건 (평가 {len(valid_small)}건)")
print(f"배치 {TRAIN_BS} × 누적 {GRAD_ACCUM} | 워커 {NUM_WORKERS} | 학습 해상도 {TRAIN_TOKENS}토큰")

---
# 추론 함수 (로짓 스코어링 + 보기 순서 TTA)

학습 루프에서 **검증 정확도**를 재는 데도 쓰므로 학습보다 먼저 정의합니다.

### 생성 대신 확률을 비교합니다
원래 코드는 모델이 글자를 **생성**하게 한 뒤 문자열에서 `a~d`를 찾고,
못 찾으면 무조건 `"a"`로 찍었습니다. 게다가 `batch_decode(out_ids)`가
**프롬프트까지 통째로 디코딩**해서 파싱이 더 불안정했습니다.

대신 `a`, `b`, `c`, `d` 네 토큰의 **확률을 직접 비교**하면 파싱 실패가 **0건**이 됩니다.

In [ ]:
CHOICE_IDS = [
    sorted({tokenizer.encode(v, add_special_tokens=False)[0] for v in (L, L.upper())})
    for L in LETTERS
]
print("선지 토큰:", dict(zip(LETTERS, CHOICE_IDS)))


@torch.no_grad()
def score_one(img, question, options):
    '''이미지 1장 + 보기 4개 -> [4] 확률 배열'''
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(question, *options)},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt")
    inputs = {k: (v.to(model.device, dtype=DTYPE) if torch.is_tensor(v) and v.dtype.is_floating_point
                  else v.to(model.device) if torch.is_tensor(v) else v)
              for k, v in inputs.items()}

    out = model.generate(**inputs, max_new_tokens=1, do_sample=False,
                         repetition_penalty=1.0,
                         output_logits=True, return_dict_in_generate=True,
                         pad_token_id=tokenizer.pad_token_id)
    logits = (out.logits[0] if getattr(out, "logits", None) else out.scores[0])[0].float()
    lp = torch.log_softmax(logits, dim=-1)
    per_letter = torch.stack([torch.logsumexp(lp[ids], dim=-1) for ids in CHOICE_IDS])
    return torch.softmax(per_letter, dim=-1).cpu().numpy()


def predict_probs(df, n_perm=None, tta_conf=None, desc="추론", resume_path=None):
    """
    원래 보기 순서 기준 확률 [N,4].

    적응형 TTA: 1회차에서 이미 확신(>= tta_conf)하면 추가 치환을 건너뛴다.
    resume_path 를 주면 중간 결과를 저장해, 런타임이 끊겨도 이어서 할 수 있다.
    """
    n_perm   = n_perm if n_perm is not None else N_PERM
    tta_conf = tta_conf if tta_conf is not None else TTA_CONF
    set_pixels(processor, MIN_TOKENS, INFER_TOKENS)

    was_training = model.training
    was_gc = bool(getattr(model, "is_gradient_checkpointing", False))
    model.eval()
    if was_gc:
        model.gradient_checkpointing_disable()
    model.config.use_cache = True

    probs = np.zeros((len(df), 4))
    start = 0
    if resume_path and os.path.exists(resume_path):          # 끊겼다면 이어서
        saved = np.load(resume_path)
        if saved["probs"].shape == probs.shape:
            probs, start = saved["probs"], int(saved["done"])
            print(f"  이전 진행분 {start}건을 불러와 이어서 진행합니다")

    n_calls = 0
    t0 = time.time()
    pbar = tqdm(range(start, len(df)), desc=desc, unit="문항", initial=start, total=len(df))
    for i in pbar:
        row = df.iloc[i]
        img = load_image(row["path"], target_tokens=INFER_TOKENS)
        options = [row["a"], row["b"], row["c"], row["d"]]

        acc = np.zeros(4)
        for s in range(n_perm):
            perm = [(k + s) % 4 for k in range(4)]     # perm[k] = k번 자리에 놓을 원래 보기
            p = score_one(img, row["question"], [options[perm[k]] for k in range(4)])
            n_calls += 1
            for k in range(4):
                acc[perm[k]] += p[k]                   # 화면 자리 -> 원래 보기로 되돌림
            # ★ 적응형: 1회차부터 확신하면 나머지 치환 생략
            if s == 0 and p.max() >= tta_conf:
                break

        probs[i] = acc / acc.sum()

        if i % 50 == 0:
            pbar.set_postfix(평균치환=f"{n_calls/max(1,i-start+1):.2f}회")
            if resume_path:
                np.savez(resume_path, probs=probs, done=i + 1)

    if resume_path:
        np.savez(resume_path, probs=probs, done=len(df))

    if was_gc:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.config.use_cache = False
    if was_training:
        model.train()
    set_pixels(processor, MIN_TOKENS, TRAIN_TOKENS)

    el = time.time() - t0
    if len(df) - start > 0:
        print(f"  {desc}: {el/60:.1f}분 | 문항당 {el/(len(df)-start):.2f}초 | "
              f"평균 치환 {n_calls/(len(df)-start):.2f}회 (최대 {n_perm})")
    return probs


def eval_accuracy(df, n_perm=1, tta_conf=1.01, desc="검증"):
    p = predict_probs(df, n_perm=n_perm, tta_conf=tta_conf, desc=desc)
    pred = [LETTERS[k] for k in p.argmax(axis=1)]
    return float(np.mean([a == b for a, b in zip(pred, df["answer"].tolist())]))


# ── 전체 소요 시간을 미리 알려줍니다 ─────────────────────────
_n = min(8, len(test_df))
_t = time.time()
_ = predict_probs(test_df.head(_n), desc="속도 측정")
_sec = (time.time() - _t) / _n
print(f"\n문항당 약 {_sec:.2f}초")
print(f"테스트 {len(test_df):,}문항 예상: 약 {_sec*len(test_df)/60:.0f}분")
if RUN_FINETUNE:
    print(f"학습 {N_TRAIN:,}건 예상: 약 {N_TRAIN*1.3/60:.0f}분  "
          f"(FAST_MODE=True 로 바꾸면 생략됩니다)")
print("\n너무 길면: TTA_CONF를 0.8로 낮추거나, INFER_TOKENS를 768로 줄이세요.")

---
# fine-tuning

### 고친 것
- **bf16 + GradScaler 조합 제거** — GradScaler는 fp16 전용입니다. GPU에 맞춰 자동 분기합니다.
- **마지막 묶음도 학습** — `step % GRAD_ACCUM == 0` 만으로는 나눠떨어지지 않는 꼬리가 버려집니다.
- **학습 대상 파라미터만 옵티마이저에** — LoRA만 넘깁니다.
- **그래디언트 클리핑 추가** — 학습 폭주 방지.
- **검증을 loss가 아니라 정확도로** — 가장 좋았던 시점을 저장했다가 복원합니다.

In [ ]:
if not RUN_FINETUNE:
    print("FAST_MODE — 파인튜닝을 건너뜁니다. (약 1시간 절약)")
    print("파인튜닝까지 하시려면 설정 셀에서 FAST_MODE = False 로 바꾸세요.")
    best_state = None
else:
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)     # ★ LoRA만

    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps*0.03), total_steps)

    # ★ bf16이면 스케일러 불필요 (GradScaler는 fp16 전용)
    scaler = torch.amp.GradScaler("cuda", enabled=not USE_BF16)
    print(f"AMP: {'bfloat16 (scaler 없음)' if USE_BF16 else 'float16 + GradScaler'} | 총 {total_steps} 스텝")

    # 학습 전 기준선 = 제로샷 (LoRA 초기값은 항등이라 사실상 원래 모델)
    best_acc = eval_accuracy(valid_small, n_perm=1, desc="검증(학습 전)")
    best_state = None
    print(f"\n[step 0] 검증 정확도 {best_acc*100:.2f}%  ← 넘어야 할 기준선\n")

    gstep = 0
    for epoch in range(EPOCHS):
        train_ds.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, seen = 0.0, 0
        n_batches = len(train_loader)
        budget_hit = False

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
        t_start = time.time()
        for step, batch in enumerate(pbar, start=1):
            batch = {k: (v.to(device, dtype=DTYPE) if torch.is_tensor(v) and v.dtype.is_floating_point
                         else v.to(device) if torch.is_tensor(v) else v)
                     for k, v in batch.items()}

            with torch.autocast("cuda", dtype=DTYPE):
                loss = model(**batch).loss / GRAD_ACCUM

            if USE_BF16:
                loss.backward()
            else:
                scaler.scale(loss).backward()

            running += loss.item() * GRAD_ACCUM; seen += 1
            del batch, loss

            # ★ 마지막 꼬리 묶음도 반드시 업데이트
            if step % GRAD_ACCUM == 0 or step == n_batches:
                if not USE_BF16:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)     # ★ 클리핑
                if USE_BF16:
                    optimizer.step(); scheduler.step()
                else:
                    prev = scaler.get_scale()
                    scaler.step(optimizer); scaler.update()
                    if scaler.get_scale() >= prev:
                        scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                gstep += 1
                pbar.set_postfix(loss=f"{running/max(1,seen):.4f}", step=gstep, best=f"{best_acc*100:.1f}%")

            # ── 20배치쯤에서 전체 예상 시간을 알려줍니다 ──
            if step == 20:
                per = (time.time() - t_start) / 20
                eta = per * n_batches / 60
                print(f"\n  배치당 {per:.2f}초 → 이 에폭 예상 {eta:.0f}분 (예산 {MAX_TRAIN_MINUTES}분)")
                if eta > MAX_TRAIN_MINUTES:
                    print(f"  ⚠️ 예산 초과 예상 → {MAX_TRAIN_MINUTES}분에서 멈추고 그때까지의 최선을 씁니다.")
                    print("     더 학습하려면 N_TRAIN을 줄이거나 TRAIN_TOKENS를 낮추거나 MAX_TRAIN_MINUTES를 늘리세요.\n")

            # ── ★ 시간 예산 초과 시 중단 (몇 시간씩 도는 사고 방지) ──
            if (time.time() - t_start) / 60 > MAX_TRAIN_MINUTES:
                print(f"\n⏹️ 시간 예산 {MAX_TRAIN_MINUTES}분 도달 — {step}/{n_batches} 배치에서 중단합니다.")
                budget_hit = True
                break

        # ── 에폭 끝: 정확도로 검증하고 가장 좋았던 시점을 저장 ──
        acc = eval_accuracy(valid_small, n_perm=1, desc=f"검증(epoch {epoch+1})")
        print(f"[Epoch {epoch+1}] train_loss {running/max(1,seen):.4f} | 검증 정확도 {acc*100:.2f}%")
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}
            print("  ⭐ 최고 기록 갱신 — 이 시점을 저장")
        else:
            print("  (개선 없음)")
        model.train()

    if best_state is not None:
        set_peft_model_state_dict(model, {k: v.to(model.device) for k, v in best_state.items()})
        print(f"\n✅ 가장 좋았던 시점으로 복원 (검증 {best_acc*100:.2f}%)")
        SAVE_DIR = "/content/qwen_vl_lora"
        model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
        print("저장:", SAVE_DIR)
    else:
        print("\n⚠️ 학습이 기준선을 넘지 못했습니다 → 다음 셀에서 제로샷이 선택됩니다")

---
# inference

- **적응형 TTA**: 1회차에서 확신(>= `TTA_CONF`)하면 나머지 치환을 건너뜁니다. 보통 절반 이하로 줄어듭니다.
- **중간 저장**: 런타임이 끊겨도 이 셀만 다시 실행하면 **중단 지점부터** 이어서 합니다.
- 파인튜닝을 했다면 제로샷과 검증 세트에서 겨뤄 **이긴 쪽으로** 제출합니다.

In [ ]:
import contextlib

# 파인튜닝을 했을 때만 제로샷과 비교합니다 (FAST_MODE면 이 비교 자체가 생략됩니다)
if best_state is not None:
    with model.disable_adapter():
        acc_zs = eval_accuracy(valid_small, n_perm=N_PERM, tta_conf=TTA_CONF, desc="제로샷 검증")
    acc_ft = eval_accuracy(valid_small, n_perm=N_PERM, tta_conf=TTA_CONF, desc="파인튜닝 검증")
    print(f"\n제로샷   : {acc_zs*100:.2f}%")
    print(f"파인튜닝 : {acc_ft*100:.2f}%")
    USE_FT = acc_ft > acc_zs + 0.01      # 확실히 이길 때만 채택
    print("→", "파인튜닝 모델로 제출합니다" if USE_FT
          else "파인튜닝이 도움이 안 됐습니다. 제로샷으로 제출합니다")
else:
    USE_FT = False
    print("파인튜닝 없이 제로샷으로 추론합니다.")

ctx = contextlib.nullcontext() if USE_FT else (
    model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext())

# ★ resume_path: 런타임이 끊겨도 이 셀만 다시 실행하면 중단 지점부터 이어서 합니다
CKPT = "/content/drive/MyDrive/vqa_test_probs.npz" if os.path.isdir("/content/drive/MyDrive") \
       else "/content/vqa_test_probs.npz"

with ctx:
    test_probs = predict_probs(test_df, n_perm=N_PERM, tta_conf=TTA_CONF,
                               desc="테스트 추론", resume_path=CKPT)

preds = [LETTERS[k] for k in test_probs.argmax(axis=1)]

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
if os.path.isdir("/content/drive/MyDrive"):
    submission.to_csv("/content/drive/MyDrive/submission.csv", index=False)   # 드라이브에도 백업

assert list(submission.columns) == ["id", "answer"]
assert submission["answer"].isin(LETTERS).all()
assert submission["id"].is_unique and len(submission) == len(test_df)

print("\n✅ Saved /content/submission.csv —", len(submission), "행")
print("예측 분포:", submission["answer"].value_counts(normalize=True).round(3).to_dict())
conf = test_probs.max(axis=1)
print(f"평균 확신도 {conf.mean():.3f} | 0.5 미만 {int((conf<0.5).sum())}건 ({(conf<0.5).mean()*100:.1f}%)")
print(submission.head().to_string(index=False))

---
# 더 올리고 싶다면 (효과 순)

1. **`INFER_TOKENS`를 1024 → 1280 → 1600** — OOM만 안 나면 거의 항상 이득. **가성비 1위**
2. **`MODEL_ID`를 7B 또는 Qwen3-VL-8B로** — 설정 셀의 주석을 푸세요 (`PX`는 자동으로 맞춰집니다)
3. **`N_TRAIN`을 4000 이상으로** — 시간이 허락하면
4. **`N_PERM`은 4 유지** — 1로 줄이면 보통 1~3%p 손해

### OOM이 나면
`TRAIN_TOKENS`를 512 → 384 → 256 순으로 낮추세요. 그래도 안 되면 `INFER_TOKENS`도 낮춥니다.